In [0]:
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from delta.tables import DeltaTable 

In [0]:
weather_hist= spark.table("dbw_routemind_euskadi_dev.silver.weather_history_cleaned")
airquality_hist = spark.table("dbw_routemind_euskadi_dev.silver.air_quality_history_cleaned")
meteorology_stations = spark.table("dbw_routemind_euskadi_dev.silver.meteorology_stations")


In [0]:
# this aggregation process for weather history data is made for granulation of daily data
weather_hist_daily = weather_hist.groupBy("date","sensorId").agg(
    F.round(F.max("temperature"), 2).alias("temp_max"),
    F.round(F.min("temperature"), 2).alias("temp_min"),
    F.round(F.avg("temperature"), 2).alias("temp_avg"),
    F.round(F.avg("precipitation"),2).alias("precip_avg")
)

weather_hist_daily = weather_hist_daily.orderBy("date","sensorId")


In [0]:
# this aggregation give an average of weather metrics by municipality and date, adding geographic information
#to data 

df_stations = meteorology_stations.select("sensorId", "municipality","county")

union_weather_hist_stations = weather_hist_daily.join(df_stations, on="sensorId", how="inner")

df_weather_hist_municipality = union_weather_hist_stations.groupBy("date","municipality","county").agg(
    F.round(F.avg("temp_max"), 2).alias("temp_max_mun"),
    F.round(F.avg("temp_min"), 2).alias("temp_min_mun"),
    F.round(F.avg("temp_avg"), 2).alias("temp_avg_mun"),
    F.round(F.avg("precip_avg"),2).alias("precip_avg_mun")
)


In [0]:
df_weather_hist_municipality.show(10)

In [0]:
union_airq_hist_stations = airquality_hist.join(df_stations, on="sensorId", how="inner")

aq_hist_municipality = union_airq_hist_stations.groupBy("date","municipality","county").agg(
    F.round(F.avg("measure_PM10"), 2).alias("PM_10_avg_mun"),
    F.round(F.avg("measure_PM2_5"), 2).alias("PM2_5_avg_mun")
)

In [0]:
# the final join is a left join considering that weather data is more crucial than air quality data for ML categorisation
gold_weather_features = df_weather_hist_municipality.join(aq_hist_municipality, on=["date","municipality"], how="left")
display(gold_weather_features)

In [0]:
# In order to avoid null values in case that for a specific date or municipality there is no air quality data, 
# As traditional classification algorithms in Scikit-Learn (SKLearn) do not tolerate missing values during the training phase, 
# it is essential to handle these missing values before registering the model with MLflow.

# The following are considered safe values for the air quality metrics based on European Air Quality Directive
temp_max = gold_weather_features.select(F.avg("temp_max_mun")).first()[0]
temp_min = gold_weather_features.select(F.avg("temp_min_mun")).first()[0]
temp_avg = gold_weather_features.select(F.avg("temp_avg_mun")).first()[0]

assigned_aq_values = {
    "PM10_avg_mun": 20.0,
    "PM2_5_avg_mun": 10.0,
    "precip_avg_mun":0.0,
    "temp_max_mun": temp_max,
    "temp_min_mun": temp_min,
    "temp_avg_mun": temp_avg
}

df_gold_features_clean = gold_weather_features.fillna(assigned_aq_values)


In [0]:
target_table = "dbw_routemind_euskadi_dev.gold.weather_features"
delta_path = "abfss://gold@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/weather_features/data"

if not spark.catalog.tableExists(target_table):
    print(f"Table {target_table} doesn't exist. Creating table...")
    
    df_gold_features_clean.write \
        .format("delta") \
        .option("path", delta_path) \
        .saveAsTable(target_table)
        
    print(f"table {target_table} created. rows processed: {df_gold_features_clean.count()}")

else:
    delta_target = DeltaTable.forName(spark, target_table)
    (
        delta_target.alias("t")
        .merge(
            df_gold_features_clean.alias("s"),
            "t.date = s.date AND t.municipality = s.municipality"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE completed on {target_table}. rows processed: {df_gold_features_clean.count()}")

In [0]:
%sql
select territorycode, municipalitycode from dbw_routemind_euskadi_dev.silver.visit_points
limit 10

In [0]:
%sql
select count (distinct municipality) from dbw_routemind_euskadi_dev.silver.meteorology_stations